# 🌿 Red neuronal para reconocer los tipos de hoja de coca
### ADEPCOCA · Entrenamiento de la CNN **YOLO26-cls** para la *API de Reconocimiento de Hoja de Coca*

Este notebook entrena, evalúa a fondo y empaqueta la red neuronal convolucional que usa la API (carpeta **API COCA**).
Al final descargas `modelo_coca_api.zip`, lo descomprimes dentro de la carpeta de la API y queda lista.

**Es tu propia IA: una red neuronal convolucional (CNN).** Todo el reconocimiento lo hace esta red entrenada con tus
fotos y corre en tu propio servidor. Además de clasificar el tipo de hoja, sabe decir *«esto no es una hoja de coca»*,
explica su resultado en lenguaje natural y muestra **dónde se fijó** (mapas de atención Grad-CAM).

| Paso | Qué hace |
|---|---|
| **1** Configuración | Datos de tu proyecto de Roboflow e hiperparámetros (**única celda que debes editar**) |
| **2** Entorno | Verifica la GPU, instala Ultralytics y Roboflow y conecta Google Drive |
| **3** Dataset | Descarga tu dataset etiquetado desde Roboflow |
| **4** Preparación | Limpieza, eliminación de fugas entre train/val/test, división estratificada, clase **«no es hoja de coca»** y balanceo |
| **5** Exploración | Distribución de clases y ejemplos de cada tipo de hoja |
| **6** CNN y entrenamiento | Arquitectura de la **red convolucional capa por capa** y transfer learning desde ImageNet con *early stopping* |
| **7–14** Evaluación | Curvas por época, validación oficial, TTA, métricas por clase, **matrices de confusión**, ROC/PR, calibración y umbrales |
| **15–16** Explicabilidad | Galería de errores, **mapas de atención Grad-CAM** y lo que ve cada **capa convolucional** |
| **17–18** Robustez y detector OOD | Luz, desenfoque, rotación, ruido; y fotos que no se parecen a nada del dataset |
| **19–20** Velocidad y exportación | Paquete `modelo_coca_api.zip` listo para la API |
| **21** Prueba | Analiza tus propias fotos con la misma lógica que usa la API |

**Antes de empezar**
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4` (o superior).
2. En el panel 🔑 **Secretos** (barra izquierda) agrega `ROBOFLOW_API_KEY` con tu API key privada de Roboflow
   (*Roboflow → Settings → API Keys*) y activa *Acceso del notebook*. Si no lo haces, te la pedirá al ejecutar.
3. Completa la celda de **Configuración** y ejecuta `Entorno de ejecución → Ejecutar todas`.

⏱️ Con unas 3 000 imágenes y una GPU T4, el entrenamiento completo toma entre 30 y 60 minutos.

## 1. Configuración

Los datos de Roboflow los encuentras en tu proyecto → **Versions** → **Download Dataset** → **Show download code**.
Verás algo como `rf.workspace("adepcoca").project("hojas-de-coca").version(3)`: copia esos tres valores abajo.

**Sobre el modelo:** YOLO26-cls es la red convolucional de clasificación más reciente de Ultralytics
(ImageNet top-1: s 76.0 % · m 78.1 % · **l 79.0 %** · x 79.9 %). Se usa *transfer learning*: la red ya sabe
reconocer bordes, texturas, nervaduras y colores, y aquí aprende a distinguir tus tipos de hoja.

In [ ]:
# ===================== CONFIGURACIÓN — edita solo esta celda =====================

# --- Tu dataset en Roboflow ---
ROBOFLOW_WORKSPACE = "tu-workspace"   # rf.workspace("...")
ROBOFLOW_PROYECTO = "tu-proyecto"     # .project("...")
ROBOFLOW_VERSION = 1                  # .version(N)
NUM_CLASES_ESPERADAS = 9              # los 9 tipos de hoja de coca

# --- Red neuronal (YOLO26-cls preentrenada en ImageNet) ---
MODELO_BASE = "yolo26l-cls.pt"  # yolo26s-cls.pt (rápida) · yolo26m-cls.pt · yolo26l-cls.pt (recomendada) · yolo26x-cls.pt (máxima)
IMGSZ = 320       # resolución de entrada: 224 rápida · 320 recomendada · 384 más detalle (más lenta)
EPOCAS = 100      # máximo de épocas; el early stopping detiene antes si la validación deja de mejorar
PACIENCIA = 25    # épocas sin mejorar antes de detenerse
BATCH = 32        # usa 16 si aparece "CUDA out of memory"
SEMILLA = 42      # reproducibilidad

# --- Preparación del dataset ---
PROPORCION_VAL = 0.15     # solo se usan si Roboflow no trae validación o prueba
PROPORCION_TEST = 0.15
BALANCEAR_CLASES = True   # sobre-muestreo de las clases con menos fotos (solo en entrenamiento)
FACTOR_MAX_BALANCEO = 3   # una clase puede crecer como máximo x3
MARGEN_RECORTE = 0.10     # solo datasets de detección: margen alrededor de cada caja

# --- Clase «no es hoja de coca» (la red aprende a rechazar otras cosas) ---
AGREGAR_CLASE_NO_COCA = True     # agrega ejemplos negativos: objetos, personas, animales, plantas (Caltech-101) y fondos vacíos
NOMBRE_CLASE_NO_COCA = "no_es_coca"
FRACCION_NO_COCA = 1.0           # negativos por conjunto = promedio de fotos por tipo de hoja × este factor

# --- Decisiones de la API ---
PRECISION_OBJETIVO = 0.98   # exactitud mínima para que la API declare un resultado "confirmado"
OOD_K = 10                  # vecinos del detector de fotos ajenas al dataset
OOD_PERCENTIL = 99          # % de fotos de validación que deben quedar dentro del umbral
OOD_MARGEN = 1.10           # holgura para fotos reales tomadas con otra cámara o luz
OOD_MAX_POR_CLASE = 600     # tamaño máximo del banco de referencia por clase

GUARDAR_EN_DRIVE = True     # guarda una copia del ZIP final en Google Drive (MyDrive/ADEPCOCA_modelo)

## 2. Entorno: GPU, librerías y Google Drive

In [ ]:
!nvidia-smi -L || echo "⚠️ No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU T4"

In [ ]:
%pip install -q "ultralytics>=8.4,<9" "roboflow>=1.1" scikit-learn seaborn

In [ ]:
import io, os, re, sys, copy, json, math, random, shutil, hashlib, time, platform, unicodedata, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision.transforms as T
import yaml
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from IPython.display import Image as IPyImage, Markdown, display
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, classification_report,
                             cohen_kappa_score, confusion_matrix, f1_score, log_loss, matthews_corrcoef,
                             precision_recall_curve, precision_recall_fscore_support, roc_auc_score, roc_curve,
                             top_k_accuracy_score)

import ultralytics
from ultralytics import YOLO
from ultralytics.utils import ASSETS, ASSETS_URL
from ultralytics.utils.downloads import download

warnings.filterwarnings("ignore", category=UserWarning)
EN_COLAB = "google.colab" in sys.modules
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = min(8, os.cpu_count() or 2)
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

BASE = Path("/content") if EN_COLAB else Path.cwd() / "trabajo_coca"
DIR_DESCARGA = BASE / "roboflow_dataset"
DIR_DATASET = BASE / "datasets" / "hoja_coca"
DIR_RUNS = BASE / "runs"
DIR_SALIDA = BASE / "modelo_coca_api"
DIR_PESOS = DIR_SALIDA / "weights"
DIR_REPORTE = DIR_SALIDA / "reporte"
if DIR_SALIDA.exists():
    shutil.rmtree(DIR_SALIDA)
for carpeta in (DIR_RUNS, DIR_PESOS, DIR_REPORTE):
    carpeta.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

def guardar_figura(nombre, dpi=140):
    plt.tight_layout()
    plt.savefig(DIR_REPORTE / f"{nombre}.png", dpi=dpi, bbox_inches="tight")
    plt.show()

if EN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

print(f"Python {platform.python_version()} · PyTorch {torch.__version__} · Ultralytics {ultralytics.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO DISPONIBLE (el entrenamiento será muy lento)")

## 3. Descargar el dataset desde Roboflow

* Proyecto de **clasificación** → se descarga en formato `folder` (una carpeta por tipo de hoja).
* Proyecto de **detección o segmentación** → se descarga en formato YOLO y **cada hoja etiquetada con una caja se
  recorta** como una muestra de su clase. Así la red aprende de la hoja y no del fondo.

In [ ]:
from roboflow import Roboflow

if ROBOFLOW_WORKSPACE == "tu-workspace" or ROBOFLOW_PROYECTO == "tu-proyecto":
    raise ValueError("Completa ROBOFLOW_WORKSPACE, ROBOFLOW_PROYECTO y ROBOFLOW_VERSION en la celda de Configuración (paso 1).")

def api_key_roboflow():
    try:
        from google.colab import userdata
        clave = userdata.get("ROBOFLOW_API_KEY")
        if clave:
            return clave
    except Exception:
        pass
    from getpass import getpass
    return getpass("API key privada de Roboflow (Settings → API Keys): ").strip()

rf = Roboflow(api_key=api_key_roboflow())
proyecto = rf.workspace(ROBOFLOW_WORKSPACE or None).project(ROBOFLOW_PROYECTO)
TIPO_PROYECTO = proyecto.type
FORMATO = "folder" if TIPO_PROYECTO == "classification" else "yolov8"
print(f"Proyecto: {ROBOFLOW_PROYECTO} · tipo: {TIPO_PROYECTO} · formato de descarga: {FORMATO}")

if DIR_DESCARGA.exists():
    shutil.rmtree(DIR_DESCARGA)
descarga = proyecto.version(ROBOFLOW_VERSION).download(FORMATO, location=str(DIR_DESCARGA), overwrite=True)
RUTA_DESCARGA = Path(descarga.location)
print("Descargado en:", RUTA_DESCARGA)
print("Contenido:", sorted(p.name for p in RUTA_DESCARGA.iterdir()))

## 4. Preparación del dataset

Pasos que garantizan métricas **honestas** y un modelo más preciso:

1. **Imágenes dañadas**: se descartan las que no se pueden abrir.
2. **División**: si Roboflow no trae validación o prueba, se crean por **división estratificada por grupos**
   (las copias aumentadas de una misma foto, `foto_jpg.rf.xxxx.jpg`, nunca quedan en conjuntos distintos).
3. **Fugas de datos**: una foto idéntica o casi idéntica (MD5 + *hash* perceptual) en entrenamiento y en
   validación/prueba infla las métricas; se quita de validación/prueba.
4. **Clases completas**: cada tipo de hoja debe tener ejemplos en train, val y test.
5. **Clase «no es hoja de coca»**: se agregan fotos de objetos, personas, animales y otras plantas (Caltech-101) y
   fondos vacíos, para que la red aprenda a decir *«esto no es una hoja de coca»* en lugar de forzar un tipo.
   Si tu dataset ya tiene una clase como `no_es_coca`, `otro` o `fondo`, se completa esa misma clase.
   *Mejor aún:* agrega en Roboflow una clase `no_es_coca` con fotos reales de tu entorno (otras hojas de los Yungas,
   manos, mesas, bolsas): son los ejemplos negativos más útiles.
6. **Balanceo**: las clases con menos fotos se sobre-muestrean **solo en entrenamiento** (máximo x3);
   validación y prueba nunca se tocan.

In [ ]:
EXT_IMAGEN = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ALIAS_SPLIT = {"train": "train", "valid": "val", "val": "val", "validation": "val", "test": "test"}
PRIORIDAD = {"train": 0, "val": 1, "test": 2}

def listar_imagenes(carpeta):
    carpeta = Path(carpeta)
    if not carpeta.exists():
        return []
    return sorted(p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in EXT_IMAGEN)

def grupo_de(ruta):
    # Roboflow nombra las copias aumentadas como <original>_jpg.rf.<hash>.jpg: todas forman un grupo
    return Path(ruta).name.split(".rf.")[0]

def recolectar_clasificacion(raiz):
    muestras = []
    carpetas_split = [d for d in raiz.iterdir() if d.is_dir() and d.name.lower() in ALIAS_SPLIT]
    for carpeta in carpetas_split or [raiz]:
        split = ALIAS_SPLIT.get(carpeta.name.lower()) if carpetas_split else None
        for clase_dir in sorted(d for d in carpeta.iterdir() if d.is_dir()):
            for ruta in listar_imagenes(clase_dir):
                muestras.append({"ruta": ruta, "clase": clase_dir.name, "split": split})
    return muestras

def recolectar_deteccion(raiz):
    config = yaml.safe_load((raiz / "data.yaml").read_text(encoding="utf-8"))
    nombres = config["names"]
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres, key=int)]
    muestras = []
    for carpeta in sorted(d for d in raiz.iterdir() if d.is_dir() and d.name.lower() in ALIAS_SPLIT):
        for ruta in listar_imagenes(carpeta / "images"):
            etiquetas = carpeta / "labels" / f"{ruta.stem}.txt"
            if not etiquetas.exists():
                continue
            for n, linea in enumerate(etiquetas.read_text().splitlines()):
                valores = linea.split()
                if len(valores) < 5:
                    continue
                clase, coords = int(float(valores[0])), [float(v) for v in valores[1:]]
                if len(coords) == 4:
                    cx, cy, w, h = coords
                    caja = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
                else:  # polígono de segmentación
                    caja = (min(coords[0::2]), min(coords[1::2]), max(coords[0::2]), max(coords[1::2]))
                muestras.append({"ruta": ruta, "clase": str(nombres[clase]), "split": ALIAS_SPLIT[carpeta.name.lower()],
                                 "caja": caja, "n": n})
    return muestras

def imagen_valida(ruta):
    try:
        with Image.open(ruta) as im:
            im.verify()
        with Image.open(ruta) as im:
            return min(im.convert("RGB").size) >= 32
    except Exception:
        return False

def huellas(ruta):
    md5 = hashlib.md5(Path(ruta).read_bytes()).hexdigest()
    with Image.open(ruta) as im:
        gris = np.asarray(ImageOps.exif_transpose(im).convert("L").resize((17, 16), Image.Resampling.BILINEAR), dtype=np.int16)
    dhash = np.packbits(gris[:, 1:] > gris[:, :-1]).tobytes().hex()  # hash perceptual de 256 bits
    return md5, dhash

def asignar_divisiones(muestras, p_val, p_test, semilla):
    for m in muestras:
        m["split"] = m["split"] or "train"
    faltan = [s for s in ("val", "test") if not any(m["split"] == s for m in muestras)]
    if not faltan:
        return muestras, "Se usan las divisiones de Roboflow (train / valid / test)."
    proporcion = {"val": p_val, "test": p_test}
    grupos = defaultdict(list)
    for m in muestras:
        if m["split"] == "train":
            grupos[grupo_de(m["ruta"])].append(m)
    por_clase = defaultdict(list)
    for g, ms in sorted(grupos.items()):
        por_clase[Counter(m["clase"] for m in ms).most_common(1)[0][0]].append(g)
    rng = random.Random(semilla)
    for clase, gs in sorted(por_clase.items()):
        rng.shuffle(gs)
        inicio = 0
        for s in faltan:
            n = max(1, round(len(gs) * proporcion[s])) if len(gs) >= 3 else 0
            for g in gs[inicio:inicio + n]:
                representante = sorted({m["ruta"] for m in grupos[g]}, key=str)[0]
                for m in grupos[g]:  # en val/test basta una foto por grupo (el resto son copias aumentadas)
                    m["split"] = s if m["ruta"] == representante else "descartar"
            inicio += n
    texto = ", ".join(f"{s} ({proporcion[s]:.0%})" for s in faltan)
    return [m for m in muestras if m["split"] != "descartar"], f"Roboflow no traía {texto}: se creó por división estratificada por grupos."

def eliminar_fugas(muestras):
    cache = {}
    for m in muestras:
        if m["ruta"] not in cache:
            cache[m["ruta"]] = huellas(m["ruta"])
        m["md5"], m["dhash"] = cache[m["ruta"]]
    claves = lambda m: (("md5", m["md5"]), ("dhash", m["dhash"]), ("grupo", grupo_de(m["ruta"])))
    dueno = {}
    for m in muestras:  # cada foto "pertenece" al conjunto de mayor prioridad: train > val > test
        for c in claves(m):
            if c not in dueno or PRIORIDAD[m["split"]] < PRIORIDAD[dueno[c]]:
                dueno[c] = m["split"]
    limpias, quitadas, vistos = [], Counter(), set()
    for m in muestras:
        if any(dueno[c] != m["split"] for c in claves(m)):
            quitadas[f"{m['split']} (también estaba en otro conjunto)"] += 1
            continue
        clave_dup = (m["split"], m["md5"], m.get("n"))
        if m["split"] != "train" and clave_dup in vistos:
            quitadas[f"{m['split']} (duplicada)"] += 1
            continue
        vistos.add(clave_dup)
        limpias.append(m)
    return limpias, quitadas

def garantizar_clases(muestras, semilla):
    rng = random.Random(semilla)
    clases = sorted({m["clase"] for m in muestras})
    for split in ("val", "test"):
        presentes = {m["clase"] for m in muestras if m["split"] == split}
        for clase in clases:
            if clase in presentes:
                continue
            grupos = defaultdict(list)
            for m in muestras:
                if m["split"] == "train" and m["clase"] == clase:
                    grupos[grupo_de(m["ruta"])].append(m)
            if len(grupos) < 2:
                continue
            g = rng.choice(sorted(grupos))
            representante = sorted({m["ruta"] for m in grupos[g]}, key=str)[0]
            for m in grupos[g]:
                m["split"] = split if m["ruta"] == representante else "descartar"
            print(f"  · '{clase}' no tenía fotos en {split}: se movió una desde train.")
    return [m for m in muestras if m["split"] != "descartar"]

def construir_dataset(muestras, destino, margen):
    if destino.exists():
        shutil.rmtree(destino)
    for m in muestras:
        carpeta = destino / m["split"] / m["clase"]
        carpeta.mkdir(parents=True, exist_ok=True)
        if "caja" in m:  # detección: recortar la hoja con un margen
            with Image.open(m["ruta"]) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                W, H = im.size
                x1, y1, x2, y2 = m["caja"]
                mx, my = (x2 - x1) * margen, (y2 - y1) * margen
                caja = (max(0, int((x1 - mx) * W)), max(0, int((y1 - my) * H)),
                        min(W, math.ceil((x2 + mx) * W)), min(H, math.ceil((y2 + my) * H)))
                if caja[2] - caja[0] >= 32 and caja[3] - caja[1] >= 32:
                    im.crop(caja).save(carpeta / f"{m['ruta'].stem}_obj{m['n']}.jpg", quality=95)
        else:
            destino_archivo = carpeta / m["ruta"].name
            n = 1
            while destino_archivo.exists():
                destino_archivo = carpeta / f"{m['ruta'].stem}_{n}{m['ruta'].suffix}"
                n += 1
            shutil.copy2(m["ruta"], destino_archivo)

def balancear(destino, factor_max, semilla):
    rng = random.Random(semilla)
    carpetas = sorted(d for d in (destino / "train").iterdir() if d.is_dir())
    conteo = {d.name: len(listar_imagenes(d)) for d in carpetas}
    objetivo = max(conteo.values())
    agregadas = {}
    for d in carpetas:
        archivos = listar_imagenes(d)
        rng.shuffle(archivos)
        faltan = min(objetivo - len(archivos), len(archivos) * (factor_max - 1))
        for i in range(faltan):
            origen = archivos[i % len(archivos)]
            shutil.copy2(origen, d / f"{origen.stem}__dup{i}{origen.suffix}")
        agregadas[d.name] = faltan
    return agregadas

def conteo_por_split():
    return pd.DataFrame({s: {d.name: len([p for p in listar_imagenes(d) if "__dup" not in p.stem])
                             for d in sorted((DIR_DATASET / s).iterdir()) if d.is_dir()}
                         for s in ("train", "val", "test")}).fillna(0).astype(int)

# --- Clase «no es hoja de coca» (mismos nombres que reconoce la API) ---
NOMBRES_RECHAZO = {"no_es_coca", "no_coca", "nococa", "no_es_hoja", "otro", "otros", "otra", "otras",
                   "fondo", "background", "ninguno", "ninguna", "negativo", "negativos"}

def normalizar_nombre(texto):
    sin_tildes = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", sin_tildes.lower()).strip("_")

def es_clase_rechazo(nombre):
    return normalizar_nombre(nombre) in NOMBRES_RECHAZO

def fondo_vacio(rng, ancho=640, alto=480):
    # Superficie sin hoja (mesa, papel, tela, madera): color, degradado, vetas, ruido y desenfoque al azar
    gen = np.random.default_rng(rng.randint(0, 10**6))
    color = np.array([rng.randint(60, 250) for _ in range(3)], dtype=np.float32)
    color[1] = min(color[1], color[0] + 10)  # sin tonos verdes: el verde es de las hojas
    x = np.linspace(0, 1, ancho, dtype=np.float32)[None, :, None]
    y = np.linspace(0, 1, alto, dtype=np.float32)[:, None, None]
    degradado = (rng.uniform(-40, 40) * x + rng.uniform(-40, 40) * y)
    vetas = rng.uniform(0, 18) * np.sin(2 * np.pi * (rng.uniform(2, 30) * y + rng.uniform(0, 3) * x
                                                     + 0.15 * gen.normal(0, 1, (alto, 1, 1)))) if rng.random() < 0.5 else 0
    ruido = gen.normal(0, rng.uniform(2, 12), (alto, ancho, 3))
    imagen = Image.fromarray(np.clip(color + degradado + vetas + ruido, 0, 255).astype(np.uint8))
    return imagen.filter(ImageFilter.GaussianBlur(rng.uniform(0, 2.5)))

def agregar_negativos(nombre_clase, fraccion, semilla):
    raiz = BASE / "negativos"
    if not listar_imagenes(raiz):
        download(f"{ASSETS_URL}/caltech101.zip", dir=raiz)  # 101 categorías de objetos, personas, animales y plantas
    por_categoria = defaultdict(list)
    for ruta in listar_imagenes(raiz):
        por_categoria[ruta.parent.name].append(ruta)
    if not por_categoria:
        raise RuntimeError("No se pudieron descargar las imágenes negativas (Caltech-101). Revisa la conexión de Colab.")
    rng = random.Random(semilla)
    for lista in por_categoria.values():
        rng.shuffle(lista)
    orden = []  # se intercalan las categorías: máxima variedad de objetos
    while any(por_categoria.values()):
        for categoria in sorted(por_categoria):
            if por_categoria[categoria]:
                orden.append(por_categoria[categoria].pop())

    conteo = conteo_por_split()
    hojas = [c for c in conteo.index if not es_clase_rechazo(c)]
    usadas, agregadas = 0, {}
    for split in ("train", "val", "test"):
        n = max(2, int(round(conteo.loc[hojas, split].mean() * fraccion)))
        n_fondos = max(1, n // 5)  # 20 %: superficies vacías (tomar la foto sin hoja es un error frecuente)
        carpeta = DIR_DATASET / split / nombre_clase
        carpeta.mkdir(parents=True, exist_ok=True)
        for ruta in orden[usadas:usadas + n - n_fondos]:  # sin repetir imágenes entre conjuntos
            with Image.open(ruta) as im:
                im.convert("RGB").save(carpeta / f"neg_{ruta.parent.name}_{ruta.stem}.jpg", quality=92)
        usadas += n - n_fondos
        for k in range(n_fondos):
            fondo_vacio(rng).save(carpeta / f"neg_fondo_vacio_{split}_{k}.jpg", quality=92)
        agregadas[split] = n
    return agregadas

In [ ]:
if TIPO_PROYECTO == "classification":
    muestras = recolectar_clasificacion(RUTA_DESCARGA)
elif (RUTA_DESCARGA / "data.yaml").exists():
    muestras = recolectar_deteccion(RUTA_DESCARGA)
    print("Dataset de detección: cada hoja etiquetada con una caja será una muestra recortada.")
else:
    raise RuntimeError(f"Tipo de proyecto no soportado: {TIPO_PROYECTO}")
if not muestras:
    raise RuntimeError("No se encontraron imágenes en la descarga. Revisa ROBOFLOW_PROYECTO y ROBOFLOW_VERSION.")
print(f"Muestras encontradas: {len(muestras)}")

validez = {}
for m in muestras:
    if m["ruta"] not in validez:
        validez[m["ruta"]] = imagen_valida(m["ruta"])
muestras = [m for m in muestras if validez[m["ruta"]]]
print(f"1) Imágenes dañadas descartadas: {sum(not v for v in validez.values())}")

muestras, nota = asignar_divisiones(muestras, PROPORCION_VAL, PROPORCION_TEST, SEMILLA)
print(f"2) {nota}")

muestras, quitadas = eliminar_fugas(muestras)
print("3) Fugas y duplicados quitados:", dict(quitadas) if quitadas else "ninguno ✅")

muestras = garantizar_clases(muestras, SEMILLA)
conteo = Counter((m["split"], m["clase"]) for m in muestras)
clases_todas = sorted({m["clase"] for m in muestras})
incompletas = {c: {s: conteo[(s, c)] for s in ("train", "val", "test")} for c in clases_todas
               if min(conteo[(s, c)] for s in ("train", "val", "test")) == 0
               and not (AGREGAR_CLASE_NO_COCA and es_clase_rechazo(c))}  # esa clase se completa con negativos
if incompletas:
    raise RuntimeError(f"Estas clases no tienen fotos suficientes para entrenar y evaluar: {incompletas}. "
                       "Agrega más fotos (mínimo 3 fotos distintas por clase) en Roboflow y genera una nueva versión.")
print("4) Todas las clases tienen fotos en train, val y test ✅")

construir_dataset(muestras, DIR_DATASET, MARGEN_RECORTE)

CLASE_NO_COCA = next((c for c in clases_todas if es_clase_rechazo(c)), None)
if AGREGAR_CLASE_NO_COCA:
    CLASE_NO_COCA = CLASE_NO_COCA or NOMBRE_CLASE_NO_COCA
    negativos = agregar_negativos(CLASE_NO_COCA, FRACCION_NO_COCA, SEMILLA)
    print(f"5) Clase «{CLASE_NO_COCA}»: {negativos} imágenes negativas (objetos, personas, plantas y fondos vacíos)")
elif CLASE_NO_COCA:
    print(f"5) Se usa la clase «{CLASE_NO_COCA}» de tu dataset para rechazar fotos que no son hoja de coca")
else:
    print("5) ⚠️ Sin clase «no es hoja de coca»: la API solo podrá apoyarse en el detector de fotos ajenas")

CONTEO_ORIGINAL = conteo_por_split()
if BALANCEAR_CLASES:
    agregadas = balancear(DIR_DATASET, FACTOR_MAX_BALANCEO, SEMILLA)
    print("6) Copias de balanceo agregadas en train:", {c: n for c, n in agregadas.items() if n} or "ninguna (ya estaba balanceado)")

CLASES_DATASET = sorted(d.name for d in (DIR_DATASET / "train").iterdir() if d.is_dir())
NUM_CLASES = len(CLASES_DATASET)
TIPOS_DE_HOJA = [c for c in CLASES_DATASET if not es_clase_rechazo(c)]
print(f"\nClases ({NUM_CLASES}): {CLASES_DATASET}")
if len(TIPOS_DE_HOJA) != NUM_CLASES_ESPERADAS:
    print(f"⚠️ Se esperaban {NUM_CLASES_ESPERADAS} tipos de hoja y el dataset tiene {len(TIPOS_DE_HOJA)}. Revisa las etiquetas en Roboflow.")

## 5. Exploración del dataset

In [ ]:
tabla = CONTEO_ORIGINAL.copy()
tabla["total"] = tabla.sum(axis=1)
tabla.loc["TOTAL"] = tabla.sum()
display(tabla)
desbalance = CONTEO_ORIGINAL["train"].max() / max(CONTEO_ORIGINAL["train"].min(), 1)
print(f"Relación clase mayor / menor en train: {desbalance:.1f}x" + ("  ⚠️ desbalanceado (se compensa con el balanceo)" if desbalance > 1.5 else "  ✅"))

ax = CONTEO_ORIGINAL.plot(kind="barh", stacked=True, figsize=(11, max(4, NUM_CLASES * 0.55)), color=["#2e7d32", "#f9a825", "#1565c0"])
ax.set_xlabel("Número de imágenes"); ax.set_ylabel(""); ax.set_title("Imágenes por tipo de hoja y conjunto")
ax.legend(["Entrenamiento", "Validación", "Prueba"])
guardar_figura("distribucion_clases")

In [ ]:
por_clase = 6
fig, ejes = plt.subplots(NUM_CLASES, por_clase, figsize=(por_clase * 2.1, NUM_CLASES * 2.2))
ejes = np.atleast_2d(ejes)
for fila, clase in enumerate(CLASES_DATASET):
    fotos = [p for p in listar_imagenes(DIR_DATASET / "train" / clase) if "__dup" not in p.stem]
    for col in range(por_clase):
        eje = ejes[fila, col]
        eje.axis("off")
        if col < len(fotos):
            eje.imshow(Image.open(fotos[col]).convert("RGB"))
        if col == 0:
            eje.set_title(clase, loc="left", fontsize=11, fontweight="bold")
guardar_figura("ejemplos_por_clase", dpi=80)

## 6. La red neuronal convolucional (CNN) y su entrenamiento

### 6.1 Arquitectura de la CNN

YOLO26-cls es una **red neuronal convolucional**. La foto entra como una matriz de píxeles (alto × ancho × 3 colores) y
atraviesa capas de **convolución**: cada una aprende filtros que detectan patrones cada vez más complejos
(bordes y colores → texturas → nervaduras y manchas → forma de la hoja), mientras la imagen se reduce y los canales
aumentan. Al final, la **cabeza de clasificación** entrega una probabilidad para cada tipo de hoja.

| Bloque | Qué hace |
|---|---|
| `Conv` | Convolución 3×3 + normalización (BatchNorm) + activación SiLU. Con *stride* 2 reduce la imagen a la mitad |
| `C3k2` | Bloque CSP de convoluciones con conexiones residuales: combina y refina los rasgos |
| `C2PSA` | Convoluciones con atención espacial: da más peso a las zonas importantes de la hoja |
| `Classify` | Convolución 1×1 → promedio global → capa lineal → *softmax*: una probabilidad por clase |

La red parte con los filtros aprendidos en ImageNet (1,2 millones de fotos) y en el entrenamiento se ajustan a tus
hojas; la última capa se reemplaza por una con tus clases.

In [ ]:
BLOQUES_CNN = {
    "Conv": "Convolución 3×3 + BatchNorm + SiLU",
    "C3k2": "Convoluciones CSP residuales",
    "C2PSA": "Convoluciones + atención espacial",
    "Classify": "Conv 1×1 → promedio global → lineal → softmax",
}
cnn = YOLO(MODELO_BASE)
formas = {}
ganchos = [capa.register_forward_hook(
               lambda _m, _e, salida, i=i: formas.__setitem__(i, tuple((salida[0] if isinstance(salida, (tuple, list)) else salida).shape)))
           for i, capa in enumerate(cnn.model.model)]
with torch.no_grad():
    cnn.model.eval()(torch.zeros(1, 3, IMGSZ, IMGSZ))
for gancho in ganchos:
    gancho.remove()

ARQUITECTURA = pd.DataFrame([
    {"Capa": i, "Bloque": type(capa).__name__, "Qué hace": BLOQUES_CNN.get(type(capa).__name__, ""),
     "Salida (canales×alto×ancho)": "×".join(map(str, formas.get(i, ())[1:])),
     "Parámetros": sum(p.numel() for p in capa.parameters())}
    for i, capa in enumerate(cnn.model.model)
])
ARQUITECTURA.to_csv(DIR_REPORTE / "arquitectura_cnn.csv", index=False)
display(ARQUITECTURA.style.format({"Parámetros": "{:,}"}).hide(axis="index"))
total = int(ARQUITECTURA["Parámetros"].sum())
print(f"CNN {Path(MODELO_BASE).stem}: {len(ARQUITECTURA)} capas · {total / 1e6:.1f} millones de parámetros · "
      f"entrada {IMGSZ}×{IMGSZ}×3 · la salida de la última capa pasa de 1000 clases (ImageNet) a {NUM_CLASES} (tus clases)")
del cnn

### 6.2 Entrenamiento (transfer learning con early stopping)

| Elemento | Valor | Por qué |
|---|---|---|
| Optimizador | AdamW + *warmup* de 3 épocas + LR coseno | Ajuste fino estable de una red preentrenada |
| Tasa de aprendizaje | `0.002·5/(4+clases)` | La misma regla que usa Ultralytics para conjuntos pequeños |
| Regularización | Dropout 0.2 en la cabeza + *weight decay* | Reduce el sobreajuste |
| Aumentos | Volteo horizontal y vertical, recorte aleatorio (60–100 %), RandAugment, borrado aleatorio | La hoja puede fotografiarse en cualquier orientación y con distinta luz |
| Épocas | Hasta `EPOCAS`, con *early stopping* (`PACIENCIA`) | Se guarda `best.pt`: la época con mejor exactitud en validación |

> Si Colab se desconecta a mitad del entrenamiento y la sesión sigue activa, puedes continuar con
> `YOLO(str(DIR_RUNS / "coca_cls/weights/last.pt")).train(resume=True)`.

In [ ]:
LR0 = round(0.002 * 5 / (4 + NUM_CLASES), 6)
modelo = YOLO(MODELO_BASE)
inicio = time.time()
modelo.train(
    data=str(DIR_DATASET), epochs=EPOCAS, patience=PACIENCIA, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
    optimizer="AdamW", lr0=LR0, lrf=0.01, cos_lr=True, warmup_epochs=3, weight_decay=0.0005, dropout=0.2,
    fliplr=0.5, flipud=0.5, scale=0.4, auto_augment="randaugment", erasing=0.25,
    seed=SEMILLA, deterministic=True, project=str(DIR_RUNS), name="coca_cls", exist_ok=True, plots=True,
)
MINUTOS = (time.time() - inicio) / 60
DIR_ENTRENAMIENTO = Path(modelo.trainer.save_dir)
RUTA_MEJOR = Path(modelo.trainer.best)
MEJOR = YOLO(str(RUTA_MEJOR))
print(f"\n✅ Entrenamiento terminado en {MINUTOS:.1f} min · mejor modelo: {RUTA_MEJOR}")

## 7. Curvas de entrenamiento por época

* **Pérdida**: la de validación debe bajar junto con la de entrenamiento. Si la de entrenamiento sigue bajando y la
  de validación sube, hay **sobreajuste** (el *early stopping* lo corta y se conserva la mejor época).
* **Exactitud top-1**: porcentaje de imágenes de validación clasificadas correctamente en cada época.

In [ ]:
resultados_epocas = pd.read_csv(DIR_ENTRENAMIENTO / "results.csv")
resultados_epocas.columns = [c.strip() for c in resultados_epocas.columns]
col_top1 = next(c for c in resultados_epocas.columns if "top1" in c)
col_top5 = next(c for c in resultados_epocas.columns if "top5" in c)
col_lr = next((c for c in resultados_epocas.columns if c.startswith("lr")), None)
epocas = resultados_epocas["epoch"]
aptitud = (resultados_epocas[col_top1] + resultados_epocas[col_top5]) / 2  # criterio de Ultralytics para best.pt
MEJOR_EPOCA = int(epocas[aptitud[aptitud == aptitud.max()].index[-1]])

fig, ejes = plt.subplots(1, 3, figsize=(18, 4.8))
ejes[0].plot(epocas, resultados_epocas["train/loss"], label="Entrenamiento", color="#2e7d32", lw=2)
ejes[0].plot(epocas, resultados_epocas["val/loss"], label="Validación", color="#c62828", lw=2)
ejes[0].set(title="Pérdida (entropía cruzada)", xlabel="Época", ylabel="Pérdida")
ejes[1].plot(epocas, resultados_epocas[col_top1] * 100, color="#1565c0", lw=2, label="Top-1")
ejes[1].plot(epocas, resultados_epocas[col_top5] * 100, color="#90caf9", lw=2, ls="--", label="Top-5")
ejes[1].set(title="Exactitud en validación", xlabel="Época", ylabel="%")
for eje in ejes[:2]:
    eje.axvline(MEJOR_EPOCA, color="#f9a825", ls=":", lw=2, label=f"Mejor época ({MEJOR_EPOCA})")
    eje.legend()
if col_lr:
    ejes[2].plot(epocas, resultados_epocas[col_lr], color="#6a1b9a", lw=2)
ejes[2].set(title="Tasa de aprendizaje", xlabel="Época", ylabel="LR")
for eje in ejes:
    eje.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
guardar_figura("curvas_entrenamiento")
print(f"Épocas entrenadas: {len(resultados_epocas)} de {EPOCAS} · mejor época: {MEJOR_EPOCA} · "
      f"exactitud de validación en esa época: {resultados_epocas[col_top1].max():.2%}")

## 8. Validación oficial de Ultralytics (validación y prueba)

El conjunto de **prueba** nunca se usó para entrenar ni para elegir la mejor época: es la estimación honesta de cómo
se comportará el modelo con fotos nuevas.

In [ ]:
val_oficial = MEJOR.val(data=str(DIR_DATASET), split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
                        project=str(DIR_RUNS), name="validacion", exist_ok=True, plots=True, verbose=False)
test_oficial = MEJOR.val(data=str(DIR_DATASET), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
                         project=str(DIR_RUNS), name="prueba", exist_ok=True, plots=True, verbose=False)
print(f"Validación → top-1: {val_oficial.top1:.2%} · top-5: {val_oficial.top5:.2%}")
print(f"Prueba     → top-1: {test_oficial.top1:.2%} · top-5: {test_oficial.top5:.2%}")
for png in sorted((DIR_RUNS / "prueba").glob("confusion_matrix*normalized*.png")):
    display(IPyImage(filename=str(png), width=650))

## 9. TTA (*Test-Time Augmentation*)

La API evalúa cada foto en 4 vistas (original, espejo horizontal, espejo vertical y rotada 180°) y promedia.
Aquí se mide si conviene, **decidiéndolo con validación** (no con prueba, para no hacer trampa).

In [ ]:
recortador = T.Compose([T.Resize(IMGSZ, interpolation=T.InterpolationMode.BILINEAR), T.CenterCrop(IMGSZ)])
NOMBRES = {int(k): str(v) for k, v in MEJOR.names.items()}
CLASES = [NOMBRES[i] for i in range(len(NOMBRES))]
NC = len(CLASES)
INDICE = {c: i for i, c in enumerate(CLASES)}
IDX_RECHAZO = sorted(i for i, c in enumerate(CLASES) if es_clase_rechazo(c))
IDX_HOJAS = [i for i in range(NC) if i not in IDX_RECHAZO]

def decidir(P):
    # Misma regla que la API: primero «¿es hoja de coca?» (rechazo > 50 %); luego el tipo de hoja más probable
    P = np.atleast_2d(P)
    if not IDX_RECHAZO:
        return P.argmax(1)
    p_no = P[:, IDX_RECHAZO].sum(1)
    tipo = np.array(IDX_HOJAS)[P[:, IDX_HOJAS].argmax(1)]
    rechazo = np.array(IDX_RECHAZO)[P[:, IDX_RECHAZO].argmax(1)]
    return np.where(p_no > 0.5, rechazo, tipo)

def decision_y_confianza(P):
    P = np.atleast_2d(P)
    pred = decidir(P)
    confianza = P[np.arange(len(P)), pred]
    if IDX_RECHAZO:
        confianza = np.where(np.isin(pred, IDX_RECHAZO), P[:, IDX_RECHAZO].sum(1), confianza)
    return pred, confianza

def cargar_rgb(fuente):
    if isinstance(fuente, Image.Image):
        return fuente.convert("RGB")
    with Image.open(fuente) as im:
        return ImageOps.exif_transpose(im).convert("RGB")

def vistas_tta(im):  # las mismas 4 vistas que usa la API
    return [im, im.transpose(Image.Transpose.FLIP_LEFT_RIGHT), im.transpose(Image.Transpose.FLIP_TOP_BOTTOM),
            im.transpose(Image.Transpose.ROTATE_180)]

def predecir_probabilidades(modelo, fuentes, tta=False, transformar=None, lote=32):
    salida = []
    for i in range(0, len(fuentes), lote):
        imagenes = []
        for fuente in fuentes[i:i + lote]:
            im = cargar_rgb(fuente)
            imagenes.append(recortador(transformar(im) if transformar else im))
        vistas = [v for im in imagenes for v in (vistas_tta(im) if tta else [im])]
        resultados = modelo.predict(vistas, imgsz=IMGSZ, device=DEVICE, verbose=False)
        p = np.stack([r.probs.data.float().cpu().numpy() for r in resultados])
        salida.append(p.reshape(len(imagenes), 4, -1).mean(axis=1) if tta else p)
    return np.concatenate(salida) if salida else np.zeros((0, NC))

def extraer_embeddings(modelo, fuentes, lote=64):
    salida = []
    for i in range(0, len(fuentes), lote):
        imagenes = [recortador(cargar_rgb(f)) for f in fuentes[i:i + lote]]
        vectores = modelo.embed(imagenes, imgsz=IMGSZ, device=DEVICE, verbose=False)
        salida.append(torch.stack([v.float().flatten() for v in vectores]).cpu().numpy())
    return np.concatenate(salida)

def normalizar(E):
    return E / np.clip(np.linalg.norm(E, axis=1, keepdims=True), 1e-12, None)

def distancia_knn(E, banco, k):
    similitud = normalizar(E) @ banco.T
    return 1.0 - (-np.partition(-similitud, k - 1, axis=1)[:, k - 1])

def rutas_y_etiquetas(split):
    rutas, etiquetas = [], []
    for clase in CLASES:
        for ruta in listar_imagenes(DIR_DATASET / split / clase):
            if "__dup" not in ruta.stem:
                rutas.append(ruta)
                etiquetas.append(INDICE[clase])
    return rutas, np.array(etiquetas, dtype=int)

def calcular_ece(y, P, bins=10):
    pred, confianza = decision_y_confianza(P)
    acierto = pred == y
    ece = 0.0
    for a, b in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (confianza > a) & (confianza <= b)
        if m.any():
            ece += m.mean() * abs(acierto[m].mean() - confianza[m].mean())
    return float(ece)

def metricas(y, P):
    pred, etiquetas = decidir(P), list(range(NC))
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=etiquetas, average="macro", zero_division=0)
    Pc = np.clip(P, 1e-7, 1.0)
    Pc = Pc / Pc.sum(axis=1, keepdims=True)
    datos = {
        "exactitud": accuracy_score(y, pred), "exactitud_balanceada": balanced_accuracy_score(y, pred),
        "precision_macro": precision, "recall_macro": recall, "f1_macro": f1,
        "f1_ponderado": f1_score(y, pred, labels=etiquetas, average="weighted", zero_division=0),
        "kappa_cohen": cohen_kappa_score(y, pred), "mcc": matthews_corrcoef(y, pred),
        "log_loss": log_loss(y, Pc, labels=etiquetas), "ece": calcular_ece(y, P),
    }
    for k in (2, 3):
        if NC > k:
            datos[f"top{k}"] = top_k_accuracy_score(y, P, k=k, labels=etiquetas)
    return {clave: float(valor) for clave, valor in datos.items()}

exactitud = lambda y, P: float((decidir(P) == y).mean()) if len(y) else float("nan")

rutas_train, y_train = rutas_y_etiquetas("train")
rutas_val, y_val = rutas_y_etiquetas("val")
rutas_test, y_test = rutas_y_etiquetas("test")
print(f"Imágenes (sin copias de balanceo): train {len(y_train)} · val {len(y_val)} · test {len(y_test)}")

P_val_sin, P_val_tta = predecir_probabilidades(MEJOR, rutas_val), predecir_probabilidades(MEJOR, rutas_val, tta=True)
P_test_sin, P_test_tta = predecir_probabilidades(MEJOR, rutas_test), predecir_probabilidades(MEJOR, rutas_test, tta=True)
TTA_RECOMENDADO = exactitud(y_val, P_val_tta) >= exactitud(y_val, P_val_sin) - 0.001
P_val = P_val_tta if TTA_RECOMENDADO else P_val_sin
P_test = P_test_tta if TTA_RECOMENDADO else P_test_sin

display(pd.DataFrame({
    "Sin TTA": [exactitud(y_val, P_val_sin), exactitud(y_test, P_test_sin), metricas(y_test, P_test_sin)["log_loss"]],
    "Con TTA (4 vistas)": [exactitud(y_val, P_val_tta), exactitud(y_test, P_test_tta), metricas(y_test, P_test_tta)["log_loss"]],
}, index=["Exactitud en validación", "Exactitud en prueba", "Log-loss en prueba (menor es mejor)"]).style.format("{:.4f}"))
print("➡️ La API usará TTA:", "SÍ" if TTA_RECOMENDADO else "NO")

## 10. Métricas detalladas

* **Exactitud**: % de aciertos. **Exactitud balanceada**: promedio del acierto de cada clase (no se deja engañar por clases grandes).
* **Precisión** de una clase: de las veces que el modelo dijo esa clase, cuántas acertó.
  **Sensibilidad (recall)**: de las hojas reales de esa clase, cuántas reconoció. **F1**: equilibrio de ambas.
* **Kappa de Cohen** y **MCC**: acuerdo más allá del azar (1 = perfecto).
* **Top-2 / Top-3**: la clase correcta está entre las 2 o 3 más probables.
* **ECE**: error de calibración (cuánto se aleja el % de confianza de la exactitud real; 0 = perfecto).

In [ ]:
M_VAL, M_TEST = metricas(y_val, P_val), metricas(y_test, P_test)
NOMBRES_METRICAS = {"exactitud": "Exactitud", "exactitud_balanceada": "Exactitud balanceada",
                    "precision_macro": "Precisión (macro)", "recall_macro": "Sensibilidad (macro)", "f1_macro": "F1 (macro)",
                    "f1_ponderado": "F1 (ponderado)", "kappa_cohen": "Kappa de Cohen", "mcc": "MCC", "log_loss": "Log-loss",
                    "ece": "ECE (calibración)", "top2": "Exactitud top-2", "top3": "Exactitud top-3"}
display(pd.DataFrame({"Validación": M_VAL, "Prueba": M_TEST}).rename(index=NOMBRES_METRICAS).style.format("{:.4f}"))

reporte = classification_report(y_test, decidir(P_test), labels=list(range(NC)), target_names=CLASES, output_dict=True, zero_division=0)
df_reporte = pd.DataFrame(reporte).T.rename(columns={"precision": "Precisión", "recall": "Sensibilidad", "f1-score": "F1", "support": "Imágenes"})
df_reporte.to_csv(DIR_REPORTE / "reporte_por_clase_prueba.csv")
display(Markdown("**Reporte por clase (conjunto de prueba)**"))
display(df_reporte.style.format({"Precisión": "{:.3f}", "Sensibilidad": "{:.3f}", "F1": "{:.3f}", "Imágenes": "{:.0f}"})
        .background_gradient(subset=["F1"], cmap="Greens", vmin=0, vmax=1))

por_clase = df_reporte.loc[CLASES, ["Precisión", "Sensibilidad", "F1"]]
ax = por_clase.plot(kind="barh", figsize=(11, max(4, NC * 0.6)), color=["#1565c0", "#2e7d32", "#f9a825"], xlim=(0, 1.05))
ax.set_title("Desempeño por tipo de hoja (prueba)"); ax.set_xlabel("Valor (0 a 1)")
guardar_figura("desempeno_por_clase")

## 11. Matrices de confusión

Filas = clase real, columnas = clase predicha. La diagonal son los aciertos. La versión en % muestra qué
porcentaje de cada clase real se clasificó en cada columna (la diagonal es la sensibilidad de cada clase).

In [ ]:
def matriz_confusion(y, pred, titulo, archivo):
    cm = confusion_matrix(y, pred, labels=list(range(NC)))
    cm_pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100
    ancho = max(14, NC * 1.8)
    fig, ejes = plt.subplots(1, 2, figsize=(ancho, ancho * 0.42))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False, xticklabels=CLASES, yticklabels=CLASES,
                ax=ejes[0], linewidths=0.5, linecolor="white")
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Greens", vmin=0, vmax=100, xticklabels=CLASES, yticklabels=CLASES,
                ax=ejes[1], linewidths=0.5, linecolor="white", cbar_kws={"label": "% de la clase real"})
    for eje, sub in zip(ejes, ("número de imágenes", "% por clase real")):
        eje.set(xlabel="Clase predicha", ylabel="Clase real", title=f"{titulo} — {sub}")
        eje.tick_params(axis="x", rotation=45)
        eje.tick_params(axis="y", rotation=0)
    guardar_figura(archivo)
    pd.DataFrame(cm, index=CLASES, columns=CLASES).to_csv(DIR_REPORTE / f"{archivo}.csv")
    return cm

CM_VAL = matriz_confusion(y_val, decidir(P_val), "Validación", "matriz_confusion_validacion")
CM_TEST = matriz_confusion(y_test, decidir(P_test), "Prueba", "matriz_confusion_prueba")

confusiones = [{"Clase real": CLASES[i], "Confundida con": CLASES[j], "Imágenes": int(CM_TEST[i, j]),
                "% de la clase real": CM_TEST[i, j] / max(CM_TEST[i].sum(), 1) * 100}
               for i in range(NC) for j in range(NC) if i != j and CM_TEST[i, j] > 0]
if confusiones:
    display(Markdown("**Confusiones más frecuentes (prueba)**"))
    display(pd.DataFrame(confusiones).sort_values("Imágenes", ascending=False).head(10)
            .style.format({"% de la clase real": "{:.1f} %"}).hide(axis="index"))
else:
    print("✅ Ninguna confusión en el conjunto de prueba.")

## 12. Curvas ROC y Precisión-Recall por clase (uno contra todos)

In [ ]:
Y_test = np.eye(NC)[y_test]
colores = sns.color_palette("husl", NC)
fig, (eje_roc, eje_pr) = plt.subplots(1, 2, figsize=(16, 6.5))
AUC, AP = {}, {}
for i, clase in enumerate(CLASES):
    positivos = Y_test[:, i].sum()
    if positivos == 0 or positivos == len(Y_test):
        continue
    fpr, tpr, _ = roc_curve(Y_test[:, i], P_test[:, i])
    precision, recall, _ = precision_recall_curve(Y_test[:, i], P_test[:, i])
    AUC[clase] = roc_auc_score(Y_test[:, i], P_test[:, i])
    AP[clase] = average_precision_score(Y_test[:, i], P_test[:, i])
    eje_roc.plot(fpr, tpr, color=colores[i], lw=2, label=f"{clase} (AUC {AUC[clase]:.3f})")
    eje_pr.plot(recall, precision, color=colores[i], lw=2, label=f"{clase} (AP {AP[clase]:.3f})")
eje_roc.plot([0, 1], [0, 1], "k--", alpha=0.4)
eje_roc.set(title="Curvas ROC", xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos")
eje_pr.set(title="Curvas Precisión-Recall", xlabel="Sensibilidad (recall)", ylabel="Precisión", ylim=(0, 1.02))
eje_roc.legend(fontsize=8, loc="lower right"); eje_pr.legend(fontsize=8, loc="lower left")
guardar_figura("curvas_roc_pr")
if AUC:
    print(f"AUC macro: {np.mean(list(AUC.values())):.4f} · AP macro: {np.mean(list(AP.values())):.4f}")

## 13. Calibración de la confianza

Si el modelo dice 90 %, debería acertar ~90 % de las veces. Un modelo bien calibrado hace que el porcentaje de
confianza que muestra la API sea creíble.

In [ ]:
pred_test, conf_test = decision_y_confianza(P_test)
acierto_test = pred_test == y_test
bordes = np.linspace(0, 1, 11)
centros, exactitudes = [], []
for a, b in zip(bordes[:-1], bordes[1:]):
    m = (conf_test > a) & (conf_test <= b)
    if m.any():
        centros.append(conf_test[m].mean())
        exactitudes.append(acierto_test[m].mean())
fig, (e1, e2) = plt.subplots(1, 2, figsize=(15, 5))
e1.plot([0, 1], [0, 1], "k--", label="Calibración perfecta")
e1.plot(centros, exactitudes, "o-", color="#2e7d32", lw=2, label="Modelo")
e1.set(title=f"Diagrama de fiabilidad (ECE = {M_TEST['ece']:.3f})", xlabel="Confianza del modelo", ylabel="Exactitud real")
e1.legend()
e2.hist(conf_test[acierto_test], bins=20, range=(0, 1), alpha=0.8, color="#2e7d32", label="Aciertos")
e2.hist(conf_test[~acierto_test], bins=20, range=(0, 1), alpha=0.8, color="#c62828", label="Errores")
e2.set(title="Confianza en aciertos y errores", xlabel="Confianza", ylabel="Imágenes")
e2.legend()
guardar_figura("calibracion_confianza")

## 14. Umbrales de decisión para la API

La API combina las 5 fotos y responde con uno de estos estados:

| Estado | Condición |
|---|---|
| **confirmado** | confianza ≥ *umbral alto* y la mayoría de fotos coincide |
| **probable** | confianza ≥ *umbral mínimo* |
| **no concluyente** | confianza menor → pide repetir las fotos |

Los umbrales se eligen con **validación**: el *umbral alto* es el menor que logra `PRECISION_OBJETIVO` de exactitud,
y el *umbral mínimo* el menor que logra 90 %. Luego se comprueban en **prueba**. Se calculan solo con fotos de hojas
de coca: el rechazo de «no es hoja de coca» lo decide la propia red (más de 50 % de probabilidad).

In [ ]:
def curva_umbral(y, P, umbrales):
    hojas = ~np.isin(y, IDX_RECHAZO)  # solo fotos de hojas de coca
    pred, confianza = decision_y_confianza(P[hojas])
    acierto = pred == y[hojas]
    filas = []
    for t in umbrales:
        m = confianza >= t
        filas.append({"umbral": t, "cobertura": m.mean() if len(m) else 0.0,
                      "exactitud": acierto[m].mean() if m.any() else np.nan, "n": int(m.sum())})
    return pd.DataFrame(filas)

def elegir_umbral(curva, precision, minimo, maximo, n_min):
    validos = curva[(curva["n"] >= n_min) & (curva["exactitud"] >= precision)]
    t = float(validos["umbral"].min()) if len(validos) else maximo
    return round(float(np.clip(t, minimo, maximo)), 2), len(validos) > 0

UMBRALES = np.round(np.arange(0.30, 0.995, 0.01), 2)
curva_val, curva_test = curva_umbral(y_val, P_val, UMBRALES), curva_umbral(y_test, P_test, UMBRALES)
n_min = max(10, int(0.2 * int((~np.isin(y_val, IDX_RECHAZO)).sum())))
UMBRAL_ALTO, ok_alto = elegir_umbral(curva_val, PRECISION_OBJETIVO, 0.70, 0.95, n_min)
UMBRAL_MINIMO, _ = elegir_umbral(curva_val, 0.90, 0.40, 0.70, n_min)
UMBRAL_MINIMO = min(UMBRAL_MINIMO, round(UMBRAL_ALTO - 0.10, 2))
if not ok_alto:
    print(f"⚠️ En validación no se alcanzó {PRECISION_OBJETIVO:.0%} de exactitud con cobertura suficiente; se usa {UMBRAL_ALTO}. "
          "Más fotos de entrenamiento mejorarán esto.")

fig, eje = plt.subplots(figsize=(12, 5))
eje.plot(curva_test["umbral"], curva_test["exactitud"] * 100, color="#2e7d32", lw=2, label="Exactitud (prueba)")
eje.plot(curva_test["umbral"], curva_test["cobertura"] * 100, color="#1565c0", lw=2, label="Cobertura: fotos que superan el umbral (prueba)")
eje.axvline(UMBRAL_ALTO, color="#f9a825", ls="--", lw=2, label=f"Umbral alto = {UMBRAL_ALTO}")
eje.axvline(UMBRAL_MINIMO, color="#c62828", ls=":", lw=2, label=f"Umbral mínimo = {UMBRAL_MINIMO}")
eje.set(title="Exactitud y cobertura según el umbral de confianza", xlabel="Umbral de confianza", ylabel="%", ylim=(0, 102))
eje.legend(loc="lower left")
guardar_figura("umbrales_confianza")

hojas_test = ~np.isin(y_test, IDX_RECHAZO)
pred_hojas, conf_hojas = decision_y_confianza(P_test[hojas_test])
for nombre, t in (("Confirmado (≥ umbral alto)", UMBRAL_ALTO), ("Probable o mejor (≥ umbral mínimo)", UMBRAL_MINIMO)):
    m = conf_hojas >= t
    acierto = (pred_hojas[m] == y_test[hojas_test][m]).mean() if m.any() else float("nan")
    print(f"{nombre}: {m.mean():.1%} de las hojas de prueba, con {acierto:.2%} de exactitud")
RECHAZO_CORRECTO = RECHAZO_ERRONEO = None
if IDX_RECHAZO:
    negativas = np.isin(y_test, IDX_RECHAZO)
    RECHAZO_CORRECTO = float(np.isin(decidir(P_test[negativas]), IDX_RECHAZO).mean())
    RECHAZO_ERRONEO = float(np.isin(decidir(P_test[hojas_test]), IDX_RECHAZO).mean())
    print(f"Fotos que NO son hoja de coca rechazadas correctamente: {RECHAZO_CORRECTO:.1%}")
    print(f"Hojas de coca rechazadas por error: {RECHAZO_ERRONEO:.1%}  ← debe ser bajo")

## 15. Galería de errores (prueba)

Los errores con mayor confianza aparecen primero: revisarlos ayuda a detectar fotos mal etiquetadas en Roboflow o
tipos de hoja que necesitan más ejemplos.

In [ ]:
pred_test, conf_test = decision_y_confianza(P_test)
errores = np.where(pred_test != y_test)[0]
print(f"Errores en prueba: {len(errores)} de {len(y_test)}")
if len(errores):
    mostrar = errores[np.argsort(-conf_test[errores])][:20]
    columnas = 5
    filas = math.ceil(len(mostrar) / columnas)
    fig, ejes = plt.subplots(filas, columnas, figsize=(columnas * 3.2, filas * 3.7))
    ejes = np.atleast_1d(ejes).ravel()
    for eje in ejes:
        eje.axis("off")
    for eje, i in zip(ejes, mostrar):
        eje.imshow(cargar_rgb(rutas_test[i]))
        eje.set_title(f"Real: {CLASES[y_test[i]]}\nPredicha: {CLASES[pred_test[i]]} ({conf_test[i]:.0%})",
                      fontsize=9, color="#b71c1c")
    guardar_figura("errores_prueba", dpi=90)

## 16. ¿Cómo decide la CNN?

### 16.1 ¿Dónde mira la red? Mapas de atención (Grad-CAM)

Grad-CAM usa los gradientes de la última capa convolucional para mostrar **qué zonas de la foto hicieron que la
red eligiera esa clase** (rojo = más importante). Si la red mira la hoja (bordes, nervaduras, manchas) y no el fondo
o la mano, está aprendiendo lo correcto. La API genera estos mismos mapas para cada foto que analiza.

In [ ]:
RED_GRADCAM = copy.deepcopy(YOLO(str(RUTA_MEJOR)).model).float().eval().to("cuda:0" if torch.cuda.is_available() else "cpu")
for parametro in RED_GRADCAM.parameters():
    parametro.requires_grad_(False)

def gradcam(fuentes, indice):
    # Igual que la API: gradientes de la clase `indice` sobre la penúltima capa (C2PSA) -> mapa 0-1 por imagen
    import torch.nn.functional as F
    recortes = [recortador(cargar_rgb(f)) for f in fuentes]
    x = torch.stack([T.functional.to_tensor(r) for r in recortes]).to(next(RED_GRADCAM.parameters()).device).requires_grad_(True)
    activaciones = {}
    gancho = RED_GRADCAM.model[-2].register_forward_hook(lambda _m, _e, salida: activaciones.__setitem__("a", salida))
    try:
        with torch.enable_grad():
            salida = RED_GRADCAM(x)
            logits = salida[1] if isinstance(salida, (tuple, list)) else salida
            gradientes = torch.autograd.grad(logits[:, indice].sum(), activaciones["a"])[0]
    finally:
        gancho.remove()
    cam = torch.relu((gradientes.mean(dim=(2, 3), keepdim=True) * activaciones["a"]).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam - cam.amin(dim=(1, 2), keepdim=True)
    cam = cam / cam.amax(dim=(1, 2), keepdim=True).clamp(min=1e-8)
    return recortes, cam.detach().cpu().numpy()

def superponer(imagen, cam):
    calor = plt.get_cmap("jet")(cam)[..., :3] * 255.0
    peso = 0.25 + 0.45 * cam[..., None]
    return np.clip(np.asarray(imagen, dtype=np.float32) * (1 - peso) + calor * peso, 0, 255).astype(np.uint8)

por_clase_mapa = 3
fig, ejes = plt.subplots(NC, por_clase_mapa * 2, figsize=(por_clase_mapa * 2 * 2.2, NC * 2.4))
ejes = np.atleast_2d(ejes)
for eje in ejes.ravel():
    eje.axis("off")
for fila, clase in enumerate(CLASES):
    indices = np.where(y_test == fila)[0][:por_clase_mapa]
    if not len(indices):
        continue
    recortes, cams = gradcam([rutas_test[i] for i in indices], fila)
    for col, (recorte, cam) in enumerate(zip(recortes, cams)):
        ejes[fila, col * 2].imshow(recorte)
        ejes[fila, col * 2 + 1].imshow(superponer(recorte, cam))
    ejes[fila, 0].set_title(clase, loc="left", fontsize=11, fontweight="bold")
guardar_figura("mapas_atencion_gradcam", dpi=90)

### 16.2 ¿Qué ve cada capa convolucional?

Cada cuadro es un **canal**: la respuesta de un filtro aprendido por la CNN ante la misma hoja (se muestran los 8 más
activos de cada capa). Las primeras capas responden a bordes y colores con mucha resolución; las profundas, a
patrones abstractos (nervaduras, manchas, forma) en mapas más pequeños. Así la red construye su decisión paso a paso.

In [ ]:
hojas_prueba = np.where(~np.isin(y_test, IDX_RECHAZO))[0]
foto = recortador(cargar_rgb(rutas_test[int(hojas_prueba[0]) if len(hojas_prueba) else 0]))
CAPAS_A_VER = [i for i in (0, 2, 4, 6) if i < len(RED_GRADCAM.model) - 1]
activaciones = {}
ganchos = [RED_GRADCAM.model[i].register_forward_hook(
               lambda _m, _e, salida, i=i: activaciones.__setitem__(i, salida.detach().float().cpu()))
           for i in CAPAS_A_VER]
with torch.no_grad():
    RED_GRADCAM(T.functional.to_tensor(foto)[None].to(next(RED_GRADCAM.parameters()).device))
for gancho in ganchos:
    gancho.remove()

canales = 8
fig, ejes = plt.subplots(len(CAPAS_A_VER), canales + 1, figsize=((canales + 1) * 1.7, len(CAPAS_A_VER) * 1.9), squeeze=False)
for fila, i in enumerate(CAPAS_A_VER):
    mapa = activaciones[i][0]  # [canales, alto, ancho]
    mas_activos = mapa.abs().mean(dim=(1, 2)).argsort(descending=True)[:canales]
    ejes[fila, 0].imshow(foto)
    ejes[fila, 0].set_title(f"Capa {i} · {type(RED_GRADCAM.model[i]).__name__}\n{mapa.shape[0]} canales de {mapa.shape[1]}×{mapa.shape[2]}",
                            fontsize=7, loc="left")
    for col, canal in enumerate(mas_activos, start=1):
        ejes[fila, col].imshow(mapa[canal].numpy(), cmap="viridis")
    for eje in ejes[fila]:
        eje.axis("off")
guardar_figura("mapas_de_caracteristicas_cnn", dpi=90)

## 17. Robustez ante condiciones reales

Exactitud en prueba cuando la foto llega más oscura, más clara, borrosa, girada, con ruido o muy comprimida
(como puede pasar con la cámara del celular o de la computadora).

In [ ]:
def recomprimir_jpeg(im, calidad):
    buffer = io.BytesIO()
    im.save(buffer, format="JPEG", quality=calidad)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

def con_ruido(im, sigma=12):
    arreglo = np.asarray(im, dtype=np.float32)
    ruido = np.random.default_rng(0).normal(0, sigma, arreglo.shape)
    return Image.fromarray(np.clip(arreglo + ruido, 0, 255).astype(np.uint8))

PERTURBACIONES = {
    "Original": None,
    "Más oscura (-40 %)": lambda im: ImageEnhance.Brightness(im).enhance(0.6),
    "Más clara (+40 %)": lambda im: ImageEnhance.Brightness(im).enhance(1.4),
    "Bajo contraste": lambda im: ImageEnhance.Contrast(im).enhance(0.6),
    "Desenfoque": lambda im: im.filter(ImageFilter.GaussianBlur(2)),
    "Rotación 15°": lambda im: im.rotate(15, resample=Image.Resampling.BILINEAR, expand=True, fillcolor=(255, 255, 255)),
    "Rotación 90°": lambda im: im.rotate(90, expand=True),
    "Ruido": con_ruido,
    "JPEG calidad 30": lambda im: recomprimir_jpeg(im, 30),
}
indices = np.random.default_rng(SEMILLA).permutation(len(rutas_test))[:300]
sub_rutas, sub_y = [rutas_test[i] for i in indices], y_test[indices]
robustez = pd.DataFrame([{"Condición": nombre,
                          "Exactitud": exactitud(sub_y, predecir_probabilidades(MEJOR, sub_rutas, tta=TTA_RECOMENDADO, transformar=f))}
                         for nombre, f in PERTURBACIONES.items()])
ax = robustez.plot(kind="barh", x="Condición", y="Exactitud", legend=False, color="#2e7d32", figsize=(10, 5), xlim=(0, 1.08))
for barra, valor in zip(ax.patches, robustez["Exactitud"]):
    ax.text(barra.get_width() + 0.01, barra.get_y() + barra.get_height() / 2, f"{valor:.1%}", va="center")
ax.set_title(f"Robustez del modelo ({len(sub_y)} fotos de prueba)"); ax.set_xlabel("Exactitud"); ax.invert_yaxis()
guardar_figura("robustez")
robustez.to_csv(DIR_REPORTE / "robustez.csv", index=False)

## 18. Detector de fotos ajenas al dataset (*out-of-distribution*)

Dos protecciones trabajan juntas para que la API no invente un tipo de hoja:

1. La clase **«no es hoja de coca»** que la red aprendió en el entrenamiento.
2. Este **detector OOD**: compara cada foto con las **hojas de coca** del entrenamiento usando los *embeddings* de la
   propia red (k vecinos más cercanos, distancia coseno). Si una foto está más lejos que el umbral (por ejemplo, ruido,
   una pared o una imagen totalmente distinta), se marca como ajena y no se fuerza un tipo de hoja.

In [ ]:
rng = np.random.default_rng(SEMILLA)
seleccion = []
for c in IDX_HOJAS:  # el banco de referencia son solo hojas de coca: la pregunta es «¿se parece a una hoja?»
    idx = np.where(y_train == c)[0]
    seleccion.extend(sorted(rng.choice(idx, OOD_MAX_POR_CLASE, replace=False)) if len(idx) > OOD_MAX_POR_CLASE else idx)
E_banco = normalizar(extraer_embeddings(MEJOR, [rutas_train[i] for i in seleccion]))
y_banco = y_train[seleccion]
K_OOD = int(min(OOD_K, len(E_banco)))

d_val = distancia_knn(extraer_embeddings(MEJOR, rutas_val), E_banco, K_OOD)
d_test = distancia_knn(extraer_embeddings(MEJOR, rutas_test), E_banco, K_OOD)
hojas_val, hojas_test = ~np.isin(y_val, IDX_RECHAZO), ~np.isin(y_test, IDX_RECHAZO)
UMBRAL_OOD = float(np.percentile(d_val[hojas_val], OOD_PERCENTIL) * OOD_MARGEN)

ajenas = {
    "Autobús (Ultralytics)": cargar_rgb(ASSETS / "bus.jpg"),
    "Personas (Ultralytics)": cargar_rgb(ASSETS / "zidane.jpg"),
    "Ruido aleatorio": Image.fromarray(rng.integers(0, 255, (480, 640, 3), dtype=np.uint8)),
    "Pared gris": Image.new("RGB", (640, 480), (128, 128, 128)),
    "Cielo azul": Image.new("RGB", (640, 480), (110, 160, 230)),
}
d_ajenas = distancia_knn(extraer_embeddings(MEJOR, list(ajenas.values())), E_banco, K_OOD)
decision_ajenas = decidir(predecir_probabilidades(MEJOR, list(ajenas.values()), tta=TTA_RECOMENDADO))

def resultado_en_api(distancia, clase):
    if clase in IDX_RECHAZO:
        return "rechazada: «no es hoja de coca» ✅"
    if distancia > UMBRAL_OOD:
        return "no concluyente: ajena al dataset ✅"
    return f"⚠️ aceptada como «{CLASES[clase]}»"

print(f"Umbral OOD: {UMBRAL_OOD:.4f} (k={K_OOD}, banco de {len(E_banco)} hojas de coca)")
print(f"Hojas de coca de prueba marcadas como ajenas: {(d_test[hojas_test] > UMBRAL_OOD).mean():.1%}  ← debe ser bajo")
if (~hojas_test).any():
    print(f"Fotos de prueba que no son coca marcadas como ajenas: {(d_test[~hojas_test] > UMBRAL_OOD).mean():.1%}")
display(pd.DataFrame({
    "Imagen": list(ajenas), "Distancia": d_ajenas, "Clase según la red": [CLASES[c] for c in decision_ajenas],
    "Resultado en la API": [resultado_en_api(d, c) for d, c in zip(d_ajenas, decision_ajenas)],
}).style.format({"Distancia": "{:.4f}"}).hide(axis="index"))

fig, eje = plt.subplots(figsize=(12, 4.5))
eje.hist(d_val[hojas_val], bins=40, alpha=0.7, color="#2e7d32", label="Hojas de coca (validación)")
eje.hist(d_test[hojas_test], bins=40, alpha=0.6, color="#1565c0", label="Hojas de coca (prueba)")
if (~hojas_test).any():
    eje.hist(d_test[~hojas_test], bins=40, alpha=0.5, color="#9e9e9e", label="No es coca (prueba)")
for nombre, d in zip(ajenas, d_ajenas):
    eje.axvline(d, color="#c62828", alpha=0.6, lw=1)
eje.axvline(UMBRAL_OOD, color="black", ls="--", lw=2, label=f"Umbral = {UMBRAL_OOD:.3f}")
eje.set(title="Distancia al dataset de entrenamiento (rojo: imágenes ajenas)", xlabel="Distancia coseno al k-ésimo vecino", ylabel="Imágenes")
eje.legend()
guardar_figura("detector_ood")

np.savez_compressed(DIR_PESOS / "ood_embeddings.npz", embeddings=E_banco.astype(np.float16), etiquetas=y_banco.astype(np.int16),
                    k=np.int32(K_OOD), umbral=np.float32(UMBRAL_OOD), dimension=np.int32(E_banco.shape[1]))
print("Guardado:", DIR_PESOS / "ood_embeddings.npz")

## 19. Velocidad de inferencia

In [ ]:
muestra = [cargar_rgb(r) for r in (rutas_test or rutas_val)[:10]]

def ms_por_foto(modelo, device, tta, repeticiones=3):
    vistas = [v for im in muestra for v in (vistas_tta(recortador(im)) if tta else [recortador(im)])]
    modelo.predict(vistas[:4], imgsz=IMGSZ, device=device, verbose=False)  # calentamiento
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        modelo.predict(vistas, imgsz=IMGSZ, device=device, verbose=False)
    return (time.perf_counter() - inicio) / repeticiones / len(muestra) * 1000

filas = []
if torch.cuda.is_available():
    filas.append({"Dispositivo": torch.cuda.get_device_name(0), "ms por foto (sin TTA)": ms_por_foto(MEJOR, DEVICE, False),
                  "ms por foto (con TTA)": ms_por_foto(MEJOR, DEVICE, True)})
modelo_cpu = YOLO(str(RUTA_MEJOR))
filas.append({"Dispositivo": f"CPU ({os.cpu_count()} núcleos)", "ms por foto (sin TTA)": ms_por_foto(modelo_cpu, "cpu", False),
              "ms por foto (con TTA)": ms_por_foto(modelo_cpu, "cpu", True)})
display(pd.DataFrame(filas).style.format({"ms por foto (sin TTA)": "{:.1f}", "ms por foto (con TTA)": "{:.1f}"}).hide(axis="index"))
print("La API analiza 5 fotos por consulta; en una PC de escritorio la CPU suele ser más rápida que la de Colab.")

## 20. Exportar el paquete para la API

Se genera `modelo_coca_api.zip` con:

```
weights/best.pt               ← la red neuronal entrenada
weights/model_meta.json       ← clases, tamaño de entrada, TTA, umbrales y métricas
weights/ood_embeddings.npz    ← banco de referencia del detector de fotos ajenas
reporte/                      ← gráficos, matrices de confusión y métricas en CSV/JSON
```

In [ ]:
shutil.copy2(RUTA_MEJOR, DIR_PESOS / "best.pt")
resultados_epocas.to_csv(DIR_REPORTE / "resultados_por_epoca.csv", index=False)
for png in DIR_ENTRENAMIENTO.glob("*.png"):
    shutil.copy2(png, DIR_REPORTE / f"ultralytics_{png.name}")

CONTEO_FINAL = {s: {c: int(n) for c, n in CONTEO_ORIGINAL[s].items()} for s in CONTEO_ORIGINAL.columns}
META = {
    "proyecto": "ADEPCOCA - Reconocimiento de tipos de hoja de coca",
    "fecha_entrenamiento": datetime.now().isoformat(timespec="seconds"),
    "arquitectura": Path(MODELO_BASE).stem.replace("yolo", "YOLO"),
    "modelo_base": MODELO_BASE,
    "imgsz": IMGSZ,
    "clases": CLASES,
    "num_clases": NC,
    "clases_rechazo": [CLASES[i] for i in IDX_RECHAZO],
    "rechazo": {"correcto": RECHAZO_CORRECTO, "erroneo": RECHAZO_ERRONEO,
                "negativos_agregados": "Caltech-101 + fondos vacíos" if AGREGAR_CLASE_NO_COCA else None},
    "tta_recomendado": bool(TTA_RECOMENDADO),
    "umbrales": {"confianza_alta": UMBRAL_ALTO, "confianza_minima": UMBRAL_MINIMO, "precision_objetivo": PRECISION_OBJETIVO},
    "ood": {"k": K_OOD, "umbral": UMBRAL_OOD, "percentil": OOD_PERCENTIL, "margen": OOD_MARGEN,
            "dimension": int(E_banco.shape[1]), "banco": int(len(E_banco))},
    "metricas_validacion": M_VAL,
    "metricas_test": M_TEST,
    "entrenamiento": {"epocas_max": EPOCAS, "epocas_entrenadas": int(len(resultados_epocas)), "mejor_epoca": MEJOR_EPOCA,
                      "paciencia": PACIENCIA, "batch": BATCH, "optimizador": "AdamW", "lr0": LR0, "minutos": round(MINUTOS, 1)},
    "dataset": {"roboflow": {"workspace": ROBOFLOW_WORKSPACE, "proyecto": ROBOFLOW_PROYECTO, "version": ROBOFLOW_VERSION,
                             "tipo": TIPO_PROYECTO},
                "imagenes": CONTEO_FINAL},
    "ultralytics_version": ultralytics.__version__,
    "versiones": {"ultralytics": ultralytics.__version__, "torch": torch.__version__, "python": platform.python_version()},
}
(DIR_PESOS / "model_meta.json").write_text(json.dumps(META, indent=2, ensure_ascii=False), encoding="utf-8")
(DIR_REPORTE / "metricas.json").write_text(json.dumps({"validacion": M_VAL, "prueba": M_TEST, "auc_por_clase": AUC,
                                                       "ap_por_clase": AP, "umbrales": META["umbrales"]},
                                                      indent=2, ensure_ascii=False), encoding="utf-8")
(DIR_SALIDA / "LEEME.txt").write_text(
    "Descomprime este ZIP dentro de la carpeta 'API COCA' (se reemplaza la carpeta weights/).\n"
    "Luego reinicia la API con iniciar_api.bat y revisa http://127.0.0.1:5000/api/status\n", encoding="utf-8")

ZIP = BASE / "modelo_coca_api.zip"
if ZIP.exists():
    ZIP.unlink()
shutil.make_archive(str(ZIP.with_suffix("")), "zip", root_dir=DIR_SALIDA)
for archivo in sorted(DIR_SALIDA.rglob("*")):
    if archivo.is_file() and archivo.parent.name == "weights":
        print(f"  {archivo.relative_to(DIR_SALIDA)}  ({archivo.stat().st_size / 1e6:.1f} MB)")
print(f"\n📦 {ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB)")

if EN_COLAB and GUARDAR_EN_DRIVE:
    destino = Path("/content/drive/MyDrive/ADEPCOCA_modelo")
    destino.mkdir(parents=True, exist_ok=True)
    copia = destino / f"modelo_coca_api_{datetime.now():%Y%m%d_%H%M}.zip"
    shutil.copy2(ZIP, copia)
    print("☁️ Copia en Google Drive:", copia)
if EN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

display(Markdown(f'''
### Resumen del modelo
| | |
|---|---|
| Arquitectura | **{META["arquitectura"]}** · entrada {IMGSZ}×{IMGSZ} px · {NC} clases |
| Exactitud en prueba | **{M_TEST["exactitud"]:.2%}** (balanceada {M_TEST["exactitud_balanceada"]:.2%}) |
| F1 macro en prueba | **{M_TEST["f1_macro"]:.4f}** |
| TTA en la API | {"sí" if TTA_RECOMENDADO else "no"} |
| Umbrales | confirmado ≥ {UMBRAL_ALTO} · probable ≥ {UMBRAL_MINIMO} |
| «No es hoja de coca» | {f"rechaza {RECHAZO_CORRECTO:.1%} de lo que no es coca · rechaza por error {RECHAZO_ERRONEO:.1%} de las hojas" if IDX_RECHAZO else "sin clase de rechazo"} |
| Detector de fotos ajenas | umbral {UMBRAL_OOD:.4f} |
'''))

## 21. Prueba con tus propias fotos (misma lógica que la API)

Sube de 1 a 5 fotos de **la misma hoja** tomadas desde distintos ángulos. Se analizan como lo hace la API:
TTA, clase «no es hoja de coca», detector de fotos ajenas, promedio de las fotos, consistencia, umbrales y mapas de
atención. Si no subes fotos, se usan 5 fotos de prueba de una misma clase.

In [ ]:
def analizar_como_api(imagenes):
    P = predecir_probabilidades(MEJOR, imagenes, tta=TTA_RECOMENDADO)
    d = distancia_knn(extraer_embeddings(MEJOR, imagenes), E_banco, K_OOD)
    fuera = d > UMBRAL_OOD
    usadas = ~fuera if (~fuera).any() else np.ones_like(fuera)
    media = P[usadas].mean(axis=0)
    indice = int(decidir(media)[0])
    votos = decidir(P)
    consistencia = float((votos[usadas] == indice).mean())
    if indice in IDX_RECHAZO:
        confianza, estado = float(media[IDX_RECHAZO].sum()), "RECHAZADO: no es una hoja de coca"
    else:
        confianza = float(media[indice])
        if fuera.mean() > 0.5:
            estado = "NO CONCLUYENTE (las fotos no se parecen a las del dataset)"
        elif confianza >= UMBRAL_ALTO and consistencia >= 0.6:
            estado = "CONFIRMADO"
        elif confianza >= UMBRAL_MINIMO:
            estado = "PROBABLE"
        else:
            estado = "NO CONCLUYENTE"

    recortes, cams = gradcam(imagenes, indice)
    columnas = len(imagenes)
    fig, ejes = plt.subplots(2, columnas, figsize=(3.2 * columnas, 6.8), squeeze=False)
    for n in range(columnas):
        ejes[0, n].imshow(recortes[n])
        ejes[0, n].set_title(f"Foto {n + 1}: {CLASES[votos[n]]}" + (" · ajena" if fuera[n] else ""), fontsize=9)
        ejes[1, n].imshow(superponer(recortes[n], cams[n]))
        ejes[1, n].set_title("Dónde se fijó la red", fontsize=8)
        ejes[0, n].axis("off"); ejes[1, n].axis("off")
    plt.show()
    print(f"Resultado: {CLASES[indice]} · confianza {confianza:.1%} · consistencia {consistencia:.0%} → {estado}")
    for i in np.argsort(-media)[:3]:
        print(f"   {CLASES[i]:<25} {media[i]:6.1%}")

imagenes = []
if EN_COLAB:
    from google.colab import files
    subidas = files.upload()
    imagenes = [ImageOps.exif_transpose(Image.open(io.BytesIO(datos))).convert("RGB") for datos in subidas.values()][:5]
if not imagenes:
    clase_demo = y_test[0] if len(y_test) else 0
    imagenes = [rutas_test[i] for i in np.where(y_test == clase_demo)[0][:5]]
    print(f"Sin fotos subidas: se usan fotos de prueba de la clase '{CLASES[clase_demo]}'.")
analizar_como_api(imagenes)

## ✅ Siguientes pasos

1. Descomprime `modelo_coca_api.zip` dentro de la carpeta **API COCA** (se reemplaza `weights/`).
2. Ejecuta `iniciar_api.bat` y abre <http://127.0.0.1:5000/api/status>: debe decir `"modelo_cargado": true`
   y mostrar tus clases.
3. Completa `config/clases.json` (se crea solo al iniciar la API) con el nombre visible, la descripción y la
   recomendación de cada tipo de hoja: la API usa esos textos para redactar la explicación de cada resultado.
4. Prueba la API en <http://127.0.0.1:5000/docs> y sigue la guía `integracion_laravel/LEEME.md` para conectarla con
   el módulo de Reconocimiento de Coca de SIS-ADEPCOCA. La pantalla mostrará el resultado, la explicación de la red y
   los mapas de atención de cada foto.

**Para mejorar la precisión con el tiempo:** activa `GUARDAR_CAPTURAS=true` en la API, revisa las fotos reales que
llegan desde el sistema, súbelas a Roboflow con su etiqueta correcta, genera una nueva versión y vuelve a ejecutar
este notebook.